# Epic of Sun - Desafio 1
This jupyter notebook gather all metrics and benchmarking from all tested architectures in a single place
***

In [17]:
import json
from pathlib import Path
import pandas as pd
import re
import csv

In [2]:
# Get the directory where THIS script is located
base_path = Path.cwd()
base_path = base_path.parent  # Move one level up to the project root

In [3]:
models_path = base_path / "models"
output_path = base_path / "models" / "models_summary.csv"

In [4]:
# Load JSON files automatically

metrics_data = []
benchmarks_data = []

for model_dir in sorted(models_path.iterdir()):

    if not model_dir.is_dir():
        continue

    model_type = model_dir.name

    metrics_path = model_dir / f"{model_type}_metrics_data.json"
    benchmarks_path = model_dir / f"{model_type}_benchmarks_data.json"

    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            metrics_data.append(json.load(f))

    if benchmarks_path.exists():
        with open(benchmarks_path, "r") as f:
            benchmarks_data.append(json.load(f))

In [5]:
# Create DataFrames directly from the JSON dictionaries

metrics_df = pd.DataFrame(metrics_data)
benchmarks_df = pd.DataFrame(benchmarks_data)

In [6]:
# Remove confusion matrix

metrics_df = metrics_df.drop(columns=["confusion_matrix"], errors="ignore")

In [7]:
# Merge metrics and benchmarks

df = pd.merge(
    metrics_df,
    benchmarks_df,
    on="Model",
    how="outer"
)

In [8]:
# Use model names as columns

df = df.set_index("Model").T

In [ ]:
# Adjust Number of parameters (row) name and units to 10^6

df = df.rename(index={'Number of parameters': 'Number of parameters (10^6)'})
df.loc['Number of parameters (10^6)'] = df.loc['Number of parameters (10^6)'] / (10**6)

In [29]:
# Rename metrics

name_dict = {
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1 Score",
    "roc_auc": "ROC AUC",
    "iou": "IoU",
}
df = df.rename(index=name_dict)

In [30]:
# Check result
print(f"\nMerged DataFrame shape: {df.shape}")
print(df)


Merged DataFrame shape: (11, 6)
Model                            Attention U-Net   DeepLabV3  DeepLabV3+  \
Precision                               0.800983    0.806492    0.774237   
Recall                                  0.869888    0.816474    0.871555   
F1 Score                                0.834015    0.811453    0.820019   
ROC AUC                                 0.921232    0.911080    0.907017   
IoU                                     0.715288    0.682726    0.694942   
Number of parameters (10^6)             7.901365   25.449409   26.744961   
Parameter size (MB)                    30.141315   97.081791  102.023930   
Mean latency (ms)                      71.857162  132.083588  146.968015   
Median latency (ms)                    70.556389  130.218899  145.032465   
95th percentile latency (ms)           77.904534  145.570805  158.732555   
Latency standard deviation (ms)         4.330919    6.550102    5.752597   

Model                                  FPN      PSPNet

In [31]:
# Save

df.to_csv(output_path, index=True)
print(f"\nSaved to: {output_path}")


Saved to: /home/fdesmello/Git/spacesugar2/models/models_summary.csv


In [32]:
# Function to change the README.md file automatically with the new table

# 1. Paths to your files (change these to match your actual file names)
CSV_FILE = output_path
MARKDOWN_FILE = base_path / 'README.md'

# CHANGE THIS VALUE to control how many decimal places you want (e.g., 2 for 12.34, 1 for 12.3)
DECIMAL_PLACES = 2

def format_cell_value(val):
    """Checks if a cell contains a decimal number and formats it."""
    # If the cell is empty, just return empty text
    if not val.strip():
        return ""
    
    try:
        # Check if there is a decimal point in the data text
        if '.' in val:
            num = float(val)
            # Format the number to the exact number of decimal places selected above
            return f"{num:.{DECIMAL_PLACES}f}"
        return val
    except ValueError:
        # If it is a word (like "Laptop"), leave it as text
        return val

def update_markdown_table_from_csv():
    table_lines = []
    
    with open(CSV_FILE, 'r', encoding='utf-8') as f:
        csv_reader = csv.reader(f)
        headers = next(csv_reader, None)
        
        if not headers:
            print("Error: The CSV file is empty!")
            return
            
        table_lines.append("| " + " | ".join(headers) + " |")
        table_lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
        
        # Read each row and apply the decimal formatting rule to every single cell
        for row in csv_reader:
            formatted_row = [format_cell_value(cell) for cell in row]
            table_lines.append("| " + " | ".join(formatted_row) + " |")
            
    new_table = "\n".join(table_lines)

    with open(MARKDOWN_FILE, 'r', encoding='utf-8') as f:
        md_content = f.read()

    pattern = r"(<!-- DATA_TABLE_START -->\n)(.*?)(\n<!-- DATA_TABLE_END -->)"
    
    if not re.search(pattern, md_content, flags=re.DOTALL):
        print("Error: Could not find placeholder tags!")
        return

    updated_content = re.sub(pattern, rf"\1{new_table}\3", md_content, flags=re.DOTALL)

    with open(MARKDOWN_FILE, 'w', encoding='utf-8') as f:
        f.write(updated_content)

    print(f"Success! Table updated and decimals locked to {DECIMAL_PLACES} places.")

In [33]:
# Update the Markdown table in the README.md file
update_markdown_table_from_csv()

Success! Table updated and decimals locked to 2 places.
